In [125]:
from utils import calculate_ijk_butt_variance, create_data_with_ipc_weights, create_weibull_data
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier
import pandas as pd
import numpy as np

n = int(250)
B_RF = int(100 )
seed = 41
X_pred_point = pd.DataFrame({'X_1': [0], 'X_2': [1], 'X_3': [80], 'X_4': [40]})

In [126]:


data = create_weibull_data(params=   { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                'scale_weibull_base':   22_080       , 
                                'rate_censoring':       0.00321    , 
                                'tau': 37, 
                                'data_type':           'weibull'               ,}, random_state=42)
df_train = create_data_with_ipc_weights(params=   { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': 10,
                                'scale_weibull_base':   22_080       , 
                                'rate_censoring':       0.00321    , 
                                'tau': 37, 
                                'data_type':           'weibull'               ,},data=data)

params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)
# Prediction für X_erwartung
_ ,rf_pred = clf.predict_proba(X_pred_point.values)

df_train


,X_1,X_2,X_3,X_4,time,event,survived,weights_ipcw
0,1,1,94.332145,40.291372,51.080769,1,1,0.004343
1,0,0,80.915202,42.743694,40.876622,0,1,0.004343
2,1,1,85.807771,39.061645,26.653541,1,0,0.004235
3,0,1,79.432168,41.390719,106.693800,0,1,0.004343
4,0,1,78.295924,40.790595,15.630833,1,0,0.004156
...,...,...,...,...,...,...,...,...
245,1,1,82.825870,45.523396,34.724384,1,0,0.004298
246,0,1,87.673172,37.938316,223.656863,0,1,0.004343
247,0,1,68.598391,32.915790,89.530852,0,1,0.004343
248,0,1,68.804641,42.219064,16.180794,0,999,0.000000


In [127]:
from utils import *

calculate_ijk_wager_variance(X_pred_point=X_pred_point, clf=clf, df_train=df_train), calculate_ijk_butt_variance(X_pred_point=X_pred_point, clf=clf, df_train=df_train), calculate_ijk2_butt_variance(X_pred_point=X_pred_point, clf=clf, df_train=df_train)

(array([0.00510271]), array([0.00510271]), array([0.00510271]))

In [128]:
calculate_ijk2_butt_variance(X_pred_point=X_pred_point, clf=clf, df_train=df_train)

array([0.00510271])

In [129]:
calculate_ijk_wager_variance(X_pred_point=X_pred_point, clf=clf, df_train=df_train)

array([0.00510271])

In [19]:
T_N_b, pred = clf.predict_proba(X_pred_point.values)
N_bi = clf.nbi
df_train["weights_ipcw"] = 1 / df_train.shape[0]
weights = df_train["weights_ipcw"]
B, n = N_bi.shape
T_N_b_mean = np.mean(T_N_b, axis=0)

cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - T_N_b_mean)) / B
cov_i_hoch2 = cov_i**2
array = cov_i_hoch2/weights.values.reshape(-1,1)

biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * np.sum(weights**2)

bias_correction = n / B * np.sum(1-weights[weights > 0]) * np.var(T_N_b, axis=0, ddof=1)* np.sum(weights**2)

biased_var_estimate-bias_correction

array([0.])

In [2]:


def calculate_ijk2_butt_variance(
    clf: DecisionTreeBaggingClassifier, X_pred_point: pd.DataFrame, df_train: pd.DataFrame
) -> float:
    """
    Calculates the biased variance estimate and bias correction for a given random forest classifier,
    prediction point, and training data.
    Parameters:
    - clf: The classifier object used for prediction.
    - X_pred_point: The prediction point as a pandas DataFrame.
    - df_train: The training data as a pandas DataFrame.
    Returns:
    - biased_var_estimate: The biased variance estimate.
    - bias_correction: The bias correction.
    """

    T_N_b, pred = clf.predict_proba(X_pred_point.values)
    N_bi = clf.nbi
    weights = df_train["weights_ipcw"]
    B, n = N_bi.shape
    T_N_b_mean = np.mean(T_N_b, axis=0)

    cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - T_N_b_mean)) / B
    cov_i_hoch2 = cov_i**2
    array = cov_i_hoch2/weights.values.reshape(-1,1)**2

    biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * (1/np.sum(weights > 0)**2)

    bias_correction = n / B * (1/np.sum(weights > 0)**2) * np.var(T_N_b, axis=0, ddof=1)* np.sum(1/(weights[weights > 0]) -1)

    return biased_var_estimate-bias_correction



In [39]:
np.sum(1/(weights[weights > 0]) -1)

462.0454545454545

In [22]:
T_N_b, pred = clf.predict_proba(X_pred_point.values)
N_bi = clf.nbi
weights = df_train["weights_ipcw"]
B, n = N_bi.shape
T_N_b_mean = np.mean(T_N_b, axis=0)

cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - T_N_b_mean)) / B
cov_i_hoch2 = cov_i**2
array = cov_i_hoch2/weights.values.reshape(-1,1)

In [36]:
# zähle in weights die Werte die größer 0 sind
np.sum(weights > 0)

22